In [30]:
from pynq import Overlay, MMIO
import numpy as np
import time

# 1. Load the Bitstream

print("Loading Overlay...")
overlay = Overlay('ip_integrator.bit') 

i_bram_ctrl = overlay.bram_ctrl_iram
d_bram_ctrl = overlay.bram_ctrl_dram
reset_gpio  = overlay.axi_gpio_0      

# Make sure reset is asserted (low-active) before injecting
reset_gpio.channel1.write(0, 0)
print("Overlay loaded. RISC-V core is halted.")

Loading Overlay...
Overlay loaded. RISC-V core is halted.


In [31]:
# Configuration based on assembly and project constraints
NUM_ELEMENTS = 32
ARRAY_OFFSET = 0x040     
STATUS_OFFSET = 0x200  
MAGIC_NUM_1 = 0xCAFEBABE
MAGIC_NUM_2 = 0xDEADBEAF



# 2. Prepare and Inject Instructions 
print("Injecting hex into I-BRAM...")
with open('test-program.hex', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    # Strip whitespace and convert the hex string to a 32-bit integer
    machine_code = int(line.strip(), 16)
    # Write to BRAM. AXI BRAM Controller addresses are byte-addressed 
    i_bram_ctrl.write(i * 4, machine_code) 

# 3. Inject Test Integer Array 
print("Generating and injecting 32 random signed integers...") 
golden_array = np.random.randint(-1000, 1000, size=NUM_ELEMENTS, dtype=np.int32)
print("Array being injected:")
print(golden_array)

for i in range(NUM_ELEMENTS):
    # Convert Python signed int to 32-bit unsigned for MMIO write
    val_to_write = int(golden_array[i]) & 0xFFFFFFFF
    d_bram_ctrl.write(ARRAY_OFFSET + (i * 4), val_to_write)
    
hw_injected = []
for i in range(NUM_ELEMENTS):
        # Read back from data RAM 
        raw_val = d_bram_ctrl.read(ARRAY_OFFSET + (i * 4))
        # Convert 32-bit unsigned back to signed integer
        signed_val = (raw_val ^ 0x80000000) - 0x80000000
        hw_injected.append(signed_val)
print("Array read from BRAM before sorting:")
print(hw_injected)
        


# Reset the Status Flag "DONE" to 0 
d_bram_ctrl.write(STATUS_OFFSET, 0x00000000)
print("Injection complete.")

Injecting hex into I-BRAM...
Generating and injecting 32 random signed integers...
Array being injected:
[ 114  400 -133   45 -296  870  548  899 -252  235  676 -815  978 -377
 -300 -807 -684  880 -179    7  -83  781 -520 -776  255 -525  130  991
  389  691  -66  381]
Array read from BRAM before sorting:
[114, 400, -133, 45, -296, 870, 548, 899, -252, 235, 676, -815, 978, -377, -300, -807, -684, 880, -179, 7, -83, 781, -520, -776, 255, -525, 130, 991, 389, 691, -66, 381]
Injection complete.


In [32]:
# 4. Release Reset 
print("Releasing reset... RISC-V core is running!")
reset_gpio.channel1.write(1, 1)

# 5. Polling for Completion 
print("Polling Status Flag at 0x2000...")
start_time = time.time()
status = 0

# Constantly read the Status Flag via the AXI BRAM Controller 
while True:
    status = d_bram_ctrl.read(STATUS_OFFSET)
    if status == MAGIC_NUM_1 or status == MAGIC_NUM_2: 
        break
    
    # Optional timeout to prevent infinite hangs during demo
    if time.time() - start_time > 5.0: 
        print("TIMEOUT: RISC-V core did not complete in time.")
        break

print(f"Hardware finished! Flag read: {hex(status)}")

# 6. Retrieve Results and Automated Checking 
if status == MAGIC_NUM_1 or status == MAGIC_NUM_2:
    print("Retrieving sorted array...")
    hw_result = []
    
    for i in range(NUM_ELEMENTS):
        # Read back from data RAM 
        raw_val = d_bram_ctrl.read(ARRAY_OFFSET + (i * 4))
        signed_val = (raw_val ^ 0x80000000) - 0x80000000
        hw_result.append(signed_val)
        
    # Python Golden Result 
    golden_sorted = sorted(golden_array)
    
    # Compare
    errors = 0
    for i in range(NUM_ELEMENTS):
        if hw_result[i] != golden_sorted[i]:
            print(f"FAILURE: Mismatch at index {i}. Expected: {golden_sorted[i]}, Got: {hw_result[i]}")
            errors += 1
        else:
            print(f"SUCCESS: Match at index {i}. Expected: {golden_sorted[i]}, Got: {hw_result[i]}")
            
    if errors == 0:
        print("\n--------------------------------------")
        print("                Success!                ")
        print("\n--------------------------------------")
    else:
        print(f"\nCompleted with {errors} errors.")

Releasing reset... RISC-V core is running!
Polling Status Flag at 0x2000...
Hardware finished! Flag read: 0xcafebabe
Retrieving sorted array...
SUCCESS: Match at index 0. Expected: -815, Got: -815
SUCCESS: Match at index 1. Expected: -807, Got: -807
SUCCESS: Match at index 2. Expected: -776, Got: -776
SUCCESS: Match at index 3. Expected: -684, Got: -684
SUCCESS: Match at index 4. Expected: -525, Got: -525
SUCCESS: Match at index 5. Expected: -520, Got: -520
SUCCESS: Match at index 6. Expected: -377, Got: -377
SUCCESS: Match at index 7. Expected: -300, Got: -300
SUCCESS: Match at index 8. Expected: -296, Got: -296
SUCCESS: Match at index 9. Expected: -252, Got: -252
SUCCESS: Match at index 10. Expected: -179, Got: -179
SUCCESS: Match at index 11. Expected: -133, Got: -133
SUCCESS: Match at index 12. Expected: -83, Got: -83
SUCCESS: Match at index 13. Expected: -66, Got: -66
SUCCESS: Match at index 14. Expected: 7, Got: 7
SUCCESS: Match at index 15. Expected: 45, Got: 45
SUCCESS: Match a